# Lesson 02 - மைக்ரோசாஃப்ட் ஏஜென்ட் கட்டமைப்பை ஆராய்தல்

**Microsoft Agent Framework (MAF)** என்பது AI ஏஜென்ட்களை உருவாக்கும் ஒருங்கிணைந்த கட்டமைப்பாகும். இது நான்கு முக்கிய கட்டுமான பகுதிகளுடன் சுத்தமான, ஒருங்கிணைக்கக்கூடிய கட்டமைப்பை வழங்குகிறது:

- **Client** – AI மாதிரி முடிவுநிலை இடத்துடன் இணைத்து தொடர்பை கையாளும்
- **Agent** – வழிமுறைகள் மற்றும் கருவி வரையறைகளுடன் ஒரு கிளையண்டை சுற்றி வரம்பிடும்
- **Tools** – மாதிரி அழைக்கக்கூடிய தனிப்பயனிடப்பட்ட செயல்பாடுகளுடன் ஏஜென்ட் திறன்களை விரிவாக்கும்
- **Session** – பல சுற்று உரையாடல்களுக்கான உரையாடல் வரலாற்றை பராமரிக்கும்

இந்த பாடத்தில், இந்த கருத்துக்களை பயன்படுத்தி **பயண முன்பதிவு ஏஜென்டை** உருவாக்கப் போகிறோம், இது இலக்கு இடம் கிடைக்கும் நிலையை பரிசோதிக்கும்.


## அமைப்பு


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## முகவர் கட்டமைப்பு معماري-ஐ புரிந்துகொள்வது

Microsoft முகவர் கட்டமைப்பு ஒரு அடுக்கு معماري-ஐ பின்பற்றுகிறது:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **வாடிக்கையாளர்** – ஒரு `AzureAIProjectAgentProvider` Azure OpenAI இடைமுகத்துடன் இணைகிறது. இது அங்கீகாரம், கோரிக்கை வடிவமைப்பு மற்றும் பதில் பாகுபடுத்தலை கையாள்கிறது.
2. **முகவர்** – வாடிக்கையாளரிடமிருந்து `provider.create_agent()` மூலம் உருவாக்கப்படுகிறது, முகவர் மாதிரி அணுகலை, கட்டளைகள் (கணினி படிவம்) மற்றும் கருவிகளை ஒன்றிணைக்கும்.
3. **கருவிகள்** – `@tool` என வழிகாட்டப்பட்ட Python செயல்பாடுகள், அதை முகவர் செயல்பாடுகளை செய்ய அல்லது தரவை பெற அழைக்கலாம்.
4. **அமர்வுகள்** – ஒரு `AgentSession` பொருள் (முகவரால் `agent.create_session()` மூலம் உருவாக்கப்படுகிறது) உரையாடல் வரலாறை சேமிக்கிறது, இது பல சுற்று உரையாடலை சாத்தியமாக்குகிறது, முகவர் முந்தைய சூழலை நினைவில் வைத்திருக்கும்.

ஒவ்வொரு அடுக்கையும் படி படியாக உருவாக்குவோம்.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## @tool அலங்காரம் கொண்ட கருவிகள் சேர்த்தல்

கருவிகள் முகவர்கள் உரை உருவாக்குவதுக்கு மேலாக செயல்பட அனுமதிக்கின்றன. `@tool` அலங்காரம் சாதாரண Python செயல்பாட்டை முகவர் அழைக்கக்கூடிய ஒன்றாக மாற்றுகிறது.

முக்கிய அம்சங்கள்:
- ஒவ்வொரு அளவுருவையும் மாதிரி புரிந்து கொள்ள `Annotated[type, "description"]` பயன்படுத்தவும்.
- ஆவணக்கோவை கருவி விளக்கமாக மாறும், அது மாதிரிக்கு தெரியும்.
- `approval_mode="never_require"` என்றால், கருவி பயனர் உறுதிசெய்யாமைன்னு தானாக இயங்கும்.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## கருவிகளுடனான ஒரு முகவரியை உருவாக்குதல்

இப்போது நாங்கள் கிளையண்ட், வழிமுறைகள் மற்றும் கருவிகளை ஒருங்கிணைத்து ஒரு முகவரியாக மாற்றுகிறோம். `instructions` என்பது அமைப்பு ஊக்கம் போன்றவை — அவை முகவரியின் தனிப்பட்ட மற்றும் நடத்தையை வரையறுக்கின்றன.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## அமர்வு கொண்ட பல முறை உரையாடல்கள்

ஒரு `AgentSession` (`agent.create_session()` மூலம் உருவாக்கப்பட்டது) ஒரு உரையாடலில் உள்ள அனைத்து செய்திகளையும் கண்காணிக்கிறது. ஒவ்வொரு `agent.run()` அழைப்பிற்கும் அதே அமர்வை வழங்குவதன் மூலம், ஏஜெண்ட் முழு உரையாடல் வரலாற்றிலும் அணுகல் பெறக்கூடியது மற்றும் முந்தைய செய்திகளுக்குப் பின்னால் போக முடியும்.

ஒரு திருப்பத்தில் எங்கள் கிடைக்கும் தன்மை பCheckerை அழைக்க ஏஜெண்ட் பயன்படுத்த `tools=[check_destination_availability]` எனக் கொடுக்கிறோம்.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## சாரம்

இந்த பாடத்தில் நீங்கள் Microsoft Agent Framework ஐ உருவாக்கும் நான்கு தூண்களை ஆராய்ந்தீர்கள்:

| கருத்து | நீங்கள் கற்றுக்கொண்டது |
|---------|----------------------|
| **கிளையண்ட்** | `AzureAIProjectAgentProvider` அங்கீகார அடிப்படையிலான அத்தியாயங்களுடன் Azure OpenAI-இுடன் இணைக்கிறது |
| **எஜென்ட்** | `provider.create_agent()` ஒரு மாதிரி இணைப்பை அறிவுரைகள் மற்றும் பெயருடன் பிணைக்கிறது |
| **கருவிகள்** | `@tool` அலங்காரம் எஜென்ட் அழைக்கும் Python செயல்பாடுகளை வெளிப்படுத்துகிறது |
| **அருகல்** | `agent.create_session()` பல உரையாடல் நடவடிக்கைகளின் வரலாற்றை பராமரிக்கிறது |

இந்த கட்டமைப்புக் கூறுகள் ஒன்றிணைந்து இயற்கை உரையாடல்களை நடத்தக்கூடிய, வெளிப்புற செயல்பாடுகளை அழைக்கக்கூடிய, மற்றும் சூழல்நிலை பராமரிக்கக்கூடிய(e.g., context) எஜென்ட்களை உருவாக்குகின்றன — இது பின்னர் பாடங்களில் மேலும் மேம்பட்ட எஜென்டிக் முறைமைக்கான அடித்தளம் ஆகும்.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**எச்சரிக்கை**:  
இந்த ஆவணம் AI மொழிபெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) மூலம் மொழிபெயர்க்கப்பட்டுள்ளது. நாங்கள் துல்லியத்துக்காக முயலுகின்றோம் என்றாலும், தானாகச் செய்யப்படும் மொழிபெயர்ப்புகளில் பிழைகள் அல்லது தவறுகள் இருக்கலாம் என்பதைக் கவனத்தில் கொள்ளவும். மெய்யான ஆவணமாக உள்ள மொழியில் உள்ள அசல் ஆவணம் அதிகாரப்பூர்வ மூலமாகக் கருதப்பட வேண்டும். முக்கிய தகவல்களுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பைப் பயன்படுத்துவதால் உண்டாகும் எந்தவொரு தவறான புரிதலுக்கும் அல்லது தவறான பொருளுரைக்கு நாங்கள் பொறுப்பாக இருப்பதில்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
